In [17]:
import pandas as pd
from transformers import CLIPTokenizer, CLIPTextModel
import numpy as np
import pickle
import torch
from concurrent.futures import ThreadPoolExecutor, as_completed
import asyncio, random
from concurrent.futures import ProcessPoolExecutor
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from sklearn.metrics.pairwise import cosine_similarity

"Pierre-Auguste Renoir"
"Käthe Kollwitz"
"Max Ernst"
"Karel Appel"


In [20]:
Artist_name ="Pierre-Auguste Renoir"

In [21]:
claude_label = pd.read_excel(f"LLM Request\\comment_annotation_claude_{Artist_name.split(" ")[-1]}.xlsx")

In [22]:
claude_label.shape

(1000, 10)

In [23]:
sum(claude_label['artwork id'].duplicated())

0

In [24]:
gemini_label = pd.read_excel(f"LLM Request\\comment_annotation_gemini_{Artist_name.split(" ")[-1]}.xlsx")

In [25]:
gemini_label.shape

(1000, 10)

In [26]:
sum(gemini_label['artwork id'].duplicated())

0

In [27]:
openai_label = pd.read_excel(f"LLM Request\\comment_annotation_openai_{Artist_name.split(" ")[-1]}.xlsx")

In [28]:
openai_label.shape

(1000, 10)

In [29]:
sum(openai_label['artwork id'].duplicated())

0

In [30]:
full_df = claude_label.merge(openai_label.drop(columns=["title",'first','last','workyear from',"nationality"]),on=["artwork id"],how="outer",suffixes=("_claude","_openai"))

In [31]:
full_df.shape

(1000, 14)

In [32]:
full_df

,artwork id,title,first,last,workyear from,nationality,artistic_value_answer_claude,artistic_value_comment_claude,creativity_answer_claude,creativity_comment_claude,artistic_value_answer_openai,artistic_value_comment_openai,creativity_answer_openai,creativity_comment_openai
0,41,Baigneuse,Pierre-Auguste,Renoir,1888,French,High,Impressionist artists Berthe Morisot and Claud...,Yes,"At the beginning of the 1880s, Renoir felt he ...",High,"""Baigneuse"" by Pierre-Auguste Renoir is a nota...",Yes,"""Baigneuse"" demonstrates Renoir's creative evo..."
1,62,"Femme au peplum rouge, tête, bras",Pierre-Auguste,Renoir,1895,French,scholarly commentary on this Renoir painting f...,"To properly answer your question, I would need...",for scholarly commentary on this Renoir painti...,"To properly answer your question, I would need...",High,"""Femme au peplum rouge, tête, bras"" (1895) by ...",Yes,"In ""Femme au peplum rouge, tête, bras,"" Renoir..."
2,149,Saule au bord d'une mare,Pierre-Auguste,Renoir,1874,French,Unable to Determine,Authoritative scholarly commentary on this spe...,Unable to Determine,Without authoritative critical sources specifi...,High,"""Saule au bord d'une mare"" (Willow by a Pond) ...",Yes,"""Saule au bord d'une mare"" demonstrates Renoir..."
3,628,Compotier de fruits,Pierre-Auguste,Renoir,1890,French,"authoritative commentary on Renoir's ""Compotie...","To obtain this analysis, I recommend:**\n- Con...","for authoritative commentary on Renoir's ""Comp...","To obtain this analysis, I recommend:**\n- Con...",High,"""Compotier de fruits"" (Fruit Bowl) by Pierre-A...",Yes,"In ""Compotier de fruits,"" Renoir demonstrates ..."
4,1184,Paysage,Pierre-Auguste,Renoir,1917,French,High,"The 1917 ""Paysage"" is part of the Barnes Found...",No,While Renoir's 1917 works represent a stylisti...,High,"""Paysage"" (Landscape) by Pierre-Auguste Renoir...",Yes,"In ""Paysage,"" Renoir demonstrates a creative e..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,426721547,Le chapeau épinglé,Pierre-Auguste,Renoir,1894,French,High,The work depicts a serene view of two young wo...,Yes,Renoir treated this scene of two young girls s...,High,"""Le Chapeau Épinglé"" (The Pinned Hat) is a not...",Yes,"Renoir's ""Le Chapeau Épinglé"" demonstrates cre..."
996,426721548,Baigneuse assise,Pierre-Auguste,Renoir,1897,French,High,The work represents a culmination of Renoir's ...,Yes,The painting is composed using contradictory a...,High,"""Baigneuse assise"" (Seated Bather) by Pierre-A...",Yes,"Renoir's ""Baigneuse assise"" demonstrates a sig..."
997,426721549,Le chapeau épinglé,Pierre-Auguste,Renoir,1898,French,High,This extremely delicate work expresses the bea...,Yes,By the 1890s lithography was becoming reaccept...,High,"""Le Chapeau Épinglé"" (The Pinned Hat) is a not...",Yes,"""Le Chapeau Épinglé"" demonstrates Renoir's inn..."
998,426721550,"Étude de femme nue, assise, variante (from L'a...",Pierre-Auguste,Renoir,1904,French,scholarly commentary on this specific Renoir w...,Recommendation:** I suggest consulting:\n- The...,for scholarly commentary on this specific Reno...,Recommendation:** I suggest consulting:\n- The...,High,"""Étude de femme nue, assise, variante"" (Study ...",Yes,"Renoir's ""Étude de femme nue, assise, variante..."


In [33]:
full_df = full_df.merge(openai_label.drop(columns=["title",'first','last','workyear from',"nationality"]),on=["artwork id"],how="outer",suffixes=(None,"_openai"))

In [34]:
full_df.shape

(1000, 18)

# Embedding Convert

In [35]:
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
text_model = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32")

In [36]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = text_model.to(device)

In [37]:
def convert_one_row(model,i,texts,device):
    try:
        inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model(**inputs)
        text_embeds = outputs.pooler_output
        text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)
        text_embeds=text_embeds.cpu().numpy()
    except Exception as e:
        print(f"Error processing {i}: {e}")
        text_embeds = np.zeros([2,512])
    return i, text_embeds

In [48]:
dataset = "claude"
if dataset =="claude":
    df = claude_label.copy()
elif dataset =="gemini":
    df = gemini_label.copy()
elif dataset =="openai":
    df = openai_label.copy()

In [49]:
number_size=df.shape[0]
#number_size=10
# range_start = 30000
range_start = 0
range_end = min(range_start+number_size,df.shape[0])
N = min(number_size, df.shape[0]-range_start)

In [50]:
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Start")
embeddings = np.zeros(N, dtype=object)
with ThreadPoolExecutor(max_workers=32) as ex:
    futures = {
        ex.submit(convert_one_row, model,i, 
                  [df.iloc[i].artistic_value_comment,df.iloc[i].creativity_comment],
                  device): i
        for i in range(range_start,range_end)
    }
    for fut in as_completed(futures):
        i, text_embeds = fut.result()
        embeddings[i-range_start] =text_embeds
        if i % 500 == 0:
            print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: {i}")
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Ends")

2025-12-16 04:42:25: Start
2025-12-16 04:42:25: 0
Error processing 65: text input must be of type `str` (single example), `list[str]` (batch or single pretokenized example) or `list[list[str]]` (batch of pretokenized examples).
2025-12-16 04:42:31: 500
Error processing 892: text input must be of type `str` (single example), `list[str]` (batch or single pretokenized example) or `list[list[str]]` (batch of pretokenized examples).
Error processing 902: text input must be of type `str` (single example), `list[str]` (batch or single pretokenized example) or `list[list[str]]` (batch of pretokenized examples).
2025-12-16 04:42:38: Ends


In [51]:
np.save(f"clip_embeddings_{dataset}_{Artist_name.split(" ")[-1]}.npy", embeddings)

# Comment Check Consistency Rough

In [152]:
consistent_creative=[]
consistent_artist=[]
consistent_overall=[]
for i in range(full_df.shape[0]):
    row = full_df.iloc[i]
    if (row.artistic_value_answer_claude == row.artistic_value_answer.strip()) & (row.artistic_value_answer.strip() == row.artistic_value_answer_gemini.strip()):
        artist_con=1
        consistent_artist.append(1)
    else:
        artist_con=0
        consistent_artist.append(0)
    if (row.creativity_answer_claude == row.creativity_answer.strip()) & (row.creativity_answer.strip() == row.creativity_answer_gemini.strip()):
        creative=1
        consistent_creative.append(1)
    else:
        creative=0
        consistent_creative.append(0)

    if artist_con+creative==2:
        consistent_overall.append(1)
    else:
        consistent_overall.append(0)

In [164]:
np.sum(consistent_artist)

np.int64(78)

In [163]:
np.sum(consistent_creative)

np.int64(73)

In [153]:
np.sum(consistent_overall)

np.int64(73)

In [154]:
check=full_df.copy()
check["creative_consist"]=consistent_creative
check["artistic_consist"]=consistent_artist
check["overall_consist"]=consistent_overall

# Comment Check Consistency Hard

In [155]:
claude_embed = np.load(f"clip_embeddings_claude.npy",allow_pickle=True)

In [156]:
gemini_embed = np.load(f"clip_embeddings_gemini.npy",allow_pickle=True)

In [157]:
openai_embed = np.load(f"clip_embeddings_openai.npy",allow_pickle=True)

In [158]:
embed_consistent_creative=[]
embed_consistent_artistic=[]
embed_consistent_overall=[]
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Start")
for i in range(openai_embed.shape[0]):
    artistic_sim= cosine_similarity([claude_embed[i][0],gemini_embed[i][0],openai_embed[i][0]])
    if (artistic_sim > 0.70).all():
        artistic_con=1
        embed_consistent_artistic.append(1)
    else:
        artistic_con=0
        embed_consistent_artistic.append(0)
    creative_sim= cosine_similarity([claude_embed[i][1],gemini_embed[i][1],openai_embed[i][1]])
    if (creative_sim > 0.70).all():
        creative=1
        embed_consistent_creative.append(1)
    else:
        creative=0
        embed_consistent_creative.append(0)

    
    if artistic_con+creative==2:
        embed_consistent_overall.append(1)
    else:
        embed_consistent_overall.append(0)
    if i%2500==0:
        print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Currently at {i}")

2025-11-28 08:04:09: Start
2025-11-28 08:04:09: Currently at 0


In [159]:
np.sum(embed_consistent_overall)

np.int64(20)

In [165]:
np.sum(embed_consistent_creative)

np.int64(31)

In [160]:
check["embed_artistic_consist"]=embed_consistent_artistic
check["embed_creative_consist"]=embed_consistent_creative
check["embed_overall_consist"]=embed_consistent_overall